# 02 — Unsupervised Clustering (v2)

KMeans, Agglomerative Clustering, and Deep Embedded Clustering (DEC) over
MiniLM and RoBERTa **summary** embeddings — 6 outcomes total
(3 clusterers × 2 embedding methods), all with explicit k=16.

KMeans and Agglomerative use UMAP(n_components=50, metric="cosine") reduction
first. DEC learns its own 64-dim latent space (MLP encoder) and does not need
UMAP.

Every method saves:
- `results/metrics_{name}.json` — clustering quality metrics
- `results/cluster_labels_{name}.npy` — raw integer cluster assignments
- `results/clusters_{name}.csv` — per-cluster KeyBERT interpretation table
- `results/full_labels_{name}.csv` — every row with predicted/true label

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

import json
import numpy as np
import pandas as pd
import umap
from sklearn.cluster import KMeans, AgglomerativeClustering

from utils import config
from utils.dec import train_dec
from utils.embeddings import load_cached
from utils.interpretability import summarize_clusters
from utils.metrics import evaluate_unsupervised, hungarian_match_predictions
from utils.samples import save_full_output

train_clean = pd.read_parquet(config.PROCESSED_DIR / "train_clean.parquet")
true_labels = train_clean["label"].to_numpy()
texts = train_clean["text"].tolist()
summaries = train_clean["summary"].tolist()

suffix = "full"  # embeddings cached with this suffix (no sample cap)

METHODS = ["minilm", "roberta"]
embeddings_by_method = {}
for method in METHODS:
    arr = load_cached(f"{method}_train_{suffix}_summary")
    assert arr is not None, (
        f"Missing cached embeddings for '{method}' — run 01_embeddings.ipynb first"
    )
    embeddings_by_method[method] = arr
    print(f"Loaded {method}: shape {arr.shape}")

print(f"\nRows: {len(train_clean)} | k={config.NUM_CLASSES}")

Loaded minilm: shape (3824, 384)
Loaded roberta: shape (3824, 768)

Rows: 3824 | k=16


In [2]:
def umap_reduce(emb: np.ndarray, seed: int = config.SEED) -> np.ndarray:
    """Reduce embeddings to 50-dim cosine-space via UMAP (same params as prior notebook)."""
    reducer = umap.UMAP(n_components=50, metric="cosine", random_state=seed)
    return reducer.fit_transform(emb)

# Pre-reduce for methods that need it (KMeans, Agglomerative).
# DEC skips UMAP — it learns its own latent representation.
print("Running UMAP reduction for KMeans / Agglomerative...")
reduced_by_method = {}
for method, emb in embeddings_by_method.items():
    print(f"  UMAP {method}...")
    reduced_by_method[method] = umap_reduce(emb)
    print(f"  {method} reduced: {reduced_by_method[method].shape}")
print("UMAP done.")

Running UMAP reduction for KMeans / Agglomerative...
  UMAP minilm...


C:\Users\ACER\OneDrive\Documents\final-project\.claude\worktrees\unsupervised-dec-agglomerative\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


  minilm reduced: (3824, 50)
  UMAP roberta...


C:\Users\ACER\OneDrive\Documents\final-project\.claude\worktrees\unsupervised-dec-agglomerative\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


  roberta reduced: (3824, 50)
UMAP done.


In [3]:
config.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
all_results = {}  # method_name -> metrics dict (for display)

def run_and_save(name: str, cluster_labels: np.ndarray, emb_original: np.ndarray) -> None:
    """Evaluate, interpret, and save all artifacts for one clustering result."""
    metrics = evaluate_unsupervised(
        true_labels, cluster_labels, emb_original,
        metric_sample_size=config.SILHOUETTE_SAMPLE_SIZE, seed=config.SEED,
    )
    all_results[name] = metrics

    # Save metrics JSON
    with open(config.RESULTS_DIR / f"metrics_{name}.json", "w") as f:
        json.dump(metrics, f, indent=2)

    # Save raw cluster label array (used by notebook 09)
    np.save(config.RESULTS_DIR / f"cluster_labels_{name}.npy", cluster_labels)

    # Cluster interpretation table (KeyBERT)
    interp = summarize_clusters(
        summaries, cluster_labels, true_labels,
        emb_original,  # pass original embedding for centroid proximity, not UMAP-reduced
        config.CLASS_NAMES,
    )
    interp.to_csv(config.RESULTS_DIR / f"clusters_{name}.csv", index=False)

    # Full-row output CSV
    predicted = hungarian_match_predictions(true_labels, cluster_labels)
    save_full_output(
        texts, predicted, true_labels, config.CLASS_NAMES,
        extra_columns={"summary": summaries},
        path=config.RESULTS_DIR / f"full_labels_{name}.csv",
    )

    print(f"  {name}: ACC={metrics['ACC (Hungarian)']:.4f}  "
          f"F1={metrics['Macro F1']:.4f}  "
          f"NMI={metrics['NMI']:.4f}  "
          f"coverage={metrics['Coverage']:.2f}")


# ── KMeans ──────────────────────────────────────────────────────────────────
print("=== KMeans (k={}) ===".format(config.NUM_CLASSES))
for method in METHODS:
    emb_red = reduced_by_method[method]
    emb_orig = embeddings_by_method[method]
    km = KMeans(n_clusters=config.NUM_CLASSES, random_state=config.SEED, n_init=10)
    labels = km.fit_predict(emb_red)
    run_and_save(f"{method}_kmeans", labels, emb_orig)

# ── Agglomerative Clustering ─────────────────────────────────────────────────
print("\n=== Agglomerative Clustering (k={}, ward linkage) ===".format(config.NUM_CLASSES))
for method in METHODS:
    emb_red = reduced_by_method[method]
    emb_orig = embeddings_by_method[method]
    agg = AgglomerativeClustering(
        n_clusters=config.NUM_CLASSES,
        linkage="ward",         # minimises variance within merges; good for dense embeddings
        metric="euclidean",     # ward requires euclidean
    )
    labels = agg.fit_predict(emb_red)
    run_and_save(f"{method}_agglomerative", labels, emb_orig)

=== KMeans (k=16) ===


  minilm_kmeans: ACC=0.4482  F1=0.4443  NMI=0.4178  coverage=1.00


  roberta_kmeans: ACC=0.2440  F1=0.2351  NMI=0.2469  coverage=1.00

=== Agglomerative Clustering (k=16, ward linkage) ===


  minilm_agglomerative: ACC=0.4506  F1=0.4406  NMI=0.4041  coverage=1.00


  roberta_agglomerative: ACC=0.2432  F1=0.2335  NMI=0.2494  coverage=1.00


In [4]:
# ── Deep Embedded Clustering (DEC) ──────────────────────────────────────────
# DEC learns its own latent space — no UMAP reduction needed.
# Uses raw (pre-UMAP) MiniLM/RoBERTa embeddings directly.
print("=== Deep Embedded Clustering (DEC, k={}) ===".format(config.NUM_CLASSES))
for method in METHODS:
    emb_orig = embeddings_by_method[method]
    print(f"\n[DEC] Training on {method} embeddings (shape {emb_orig.shape})...")
    labels = train_dec(
        emb_orig,
        n_clusters=config.NUM_CLASSES,
        latent_dim=64,
        pretrain_epochs=50,
        dec_epochs=150,
        batch_size=256,
        lr_pretrain=1e-3,
        lr_dec=1e-4,
        update_interval=5,
        tol=1e-3,
        seed=config.SEED,
    )
    run_and_save(f"{method}_dec", labels, emb_orig)

=== Deep Embedded Clustering (DEC, k=16) ===

[DEC] Training on minilm embeddings (shape (3824, 384))...
[DEC] Pretraining autoencoder (50 epochs)...


  [DEC pretrain] epoch 10/50  recon_loss=0.00129


  [DEC pretrain] epoch 20/50  recon_loss=0.00107


  [DEC pretrain] epoch 30/50  recon_loss=0.00102


  [DEC pretrain] epoch 40/50  recon_loss=0.00099


...

  [DEC] epoch  135  label-change delta=0.0102


  [DEC] epoch  140  label-change delta=0.0086


  [DEC] epoch  145  label-change delta=0.0086


[DEC] Done. 16 non-empty clusters.


  roberta_dec: ACC=0.1470  F1=0.1284  NMI=0.1150  coverage=1.00


In [5]:
print("\n=== All results ===")
summary_rows = []
for name, metrics in all_results.items():
    summary_rows.append({
        "method": name,
        "ACC (Hungarian)": round(metrics["ACC (Hungarian)"], 4),
        "Macro F1": round(metrics["Macro F1"], 4),
        "NMI": round(metrics["NMI"], 4),
        "ARI": round(metrics["ARI"], 4),
        "Silhouette": round(metrics["Silhouette Score"], 4),
        "Coverage": round(metrics["Coverage"], 4),
    })
summary_df = pd.DataFrame(summary_rows).sort_values("ACC (Hungarian)", ascending=False)
print(summary_df.to_string(index=False))
print(f"\nAll artifacts saved to {config.RESULTS_DIR}")


=== All results ===
               method  ACC (Hungarian)  Macro F1    NMI    ARI  Silhouette  Coverage
 minilm_agglomerative           0.4506    0.4406 0.4041 0.2637      0.0307       1.0
        minilm_kmeans           0.4482    0.4443 0.4178 0.2840      0.0374       1.0
           minilm_dec           0.2633    0.2514 0.2346 0.1207     -0.0067       1.0
       roberta_kmeans           0.2440    0.2351 0.2469 0.1143      0.0706       1.0
roberta_agglomerative           0.2432    0.2335 0.2494 0.1147      0.0533       1.0
          roberta_dec           0.1470    0.1284 0.1150 0.0482      0.0528       1.0

All artifacts saved to C:\Users\ACER\OneDrive\Documents\final-project\.claude\worktrees\unsupervised-dec-agglomerative\results
